In [ ]:
#Finding methylation sites

In [ ]:
import os
import requests
import shutil
import pandas as pd

BASE_DIR      = r"C:\Users\jlapi\Desktop\HM450"
MANIFEST_PATH = os.path.join(BASE_DIR, "HM450_manifest.csv")
OUTPUT_CSV    = os.path.join(BASE_DIR, "CA9_probes.csv")
REGIONS_CSV   = os.path.join(BASE_DIR, "CA9_cpg_regions.csv")
MANIFEST_URL  = (
    "https://webdata.illumina.com/downloads/productfiles/"
    "humanmethylation450/humanmethylation450_15017482_v1-2.csv"
)
GENE_NAME  = "CA9"
SHORE_SIZE = 2000  # bp each side of island
SHELF_SIZE = 2000  # bp each side of shore

os.makedirs(BASE_DIR, exist_ok=True)

# Download manifest
if not os.path.exists(MANIFEST_PATH):
    print("Downloading HM450 manifest (~30 MB)...")
    with requests.get(MANIFEST_URL, stream=True) as r:
        r.raise_for_status()
        with open(MANIFEST_PATH, "wb") as f:
            shutil.copyfileobj(r.raw, f)
    print(f"Saved: {MANIFEST_PATH}")
else:
    print("Manifest already present, skipping download.")

# Find header row
skip = 0
with open(MANIFEST_PATH, "rt", encoding="latin-1") as fh:
    for line in fh:
        if line.startswith("IlmnID") or line.startswith('"IlmnID"'):
            break
        skip += 1

manifest = pd.read_csv(MANIFEST_PATH, skiprows=skip, encoding="latin-1", low_memory=False)
manifest.columns = manifest.columns.str.strip().str.replace('"', "")

# Find all CA4-annotated probes (exact gene match, chr17 only)
ca4 = manifest[
    manifest["UCSC_RefGene_Name"]
        .str.split(";")
        .apply(lambda x: isinstance(x, list) and "CA9" in x)
    & (manifest["CHR"].astype(str) == "9")
][[
    "IlmnID", "CHR", "MAPINFO",
    "UCSC_RefGene_Name", "UCSC_RefGene_Group",
    "Relation_to_UCSC_CpG_Island", "UCSC_CpG_Islands_Name"
]].sort_values("MAPINFO").reset_index(drop=True)

print(f"\nAll CpG sites annotated to {GENE_NAME}: {len(ca4)}")
print(ca4.to_string(index=False))

ca4.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved probes to: {OUTPUT_CSV}")

# Derive exact CpG region boundaries from island name
island_names = ca4["UCSC_CpG_Islands_Name"].dropna().unique()

regions = []
for island in island_names:
    try:
        chrom, coords = island.strip().split(":")
        isl_start, isl_end = [int(x) for x in coords.split("-")]
    except ValueError:
        print(f"Could not parse island name: {island}")
        continue

    regions.append(("Island",  chrom, isl_start,                              isl_end,                          island))
    regions.append(("N_Shore", chrom, isl_start - SHORE_SIZE,                 isl_start - 1,                    island))
    regions.append(("S_Shore", chrom, isl_end   + 1,                          isl_end   + SHORE_SIZE,           island))
    regions.append(("N_Shelf", chrom, isl_start - SHORE_SIZE - SHELF_SIZE,    isl_start - SHORE_SIZE - 1,       island))
    regions.append(("S_Shelf", chrom, isl_end   + SHORE_SIZE + 1,             isl_end   + SHORE_SIZE + SHELF_SIZE, island))

regions_df = pd.DataFrame(regions, columns=["Region", "CHR", "Start", "End", "Island_Name"])

print(f"\nDerived CpG region boundaries:")
print(regions_df.to_string(index=False))

regions_df.to_csv(REGIONS_CSV, index=False)
print(f"\nSaved regions to: {REGIONS_CSV}")

In [ ]:
# Location of methylation sites

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

BASE_DIR   = r"C:\Users\jlapi\Desktop\HM450"
GTF_PATH   = os.path.join(BASE_DIR, "gencode.v36.primary_assembly.annotation.gtf")
OUTPUT_FIG = os.path.join(BASE_DIR, "CA9_methylation_probes.png")

GENE_NAME         = "CA9"
TARGET_TRANSCRIPT = "ENST00000378357.9"
PROMOTER_UPSTREAM = 1000

PROBE_IDS = [
    "cg19257550", "cg20610181", "cg06908460", "cg13849253",
    "cg09566069", "cg14563831", "cg13938361",
]
MS_LABELS = {
    "cg19257550": "MS1", "cg20610181": "MS2", "cg06908460": "MS3",
    "cg13849253": "MS4", "cg09566069": "MS5", "cg14563831": "MS6",
    "cg13938361": "MS7",
}
CPG_REGIONS = [
    ("N_Shore", 35673648, 35675647, "#17becf"),
    ("Island",  35675648, 35676375, "#2ca02c"),
    ("S_Shore", 35676376, 35678375, "#9edae5"),
    ("S_Shelf", 35678376, 35680375, "#ffbb78"),
]

# Load probe positions
df = pd.read_csv(os.path.join(BASE_DIR, "CA9_probes.csv"))
df = df[df["IlmnID"].isin(PROBE_IDS)].sort_values("MAPINFO")
probes = [(row["IlmnID"], int(row["MAPINFO"])) for _, row in df.iterrows()]

# Parse GTF
exons, tss, strand = [], None, None

with open(GTF_PATH, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        cols = line.strip().split("\t")
        if len(cols) != 9:
            continue

        feature = cols[2]
        if feature not in ("exon", "transcript"):
            continue

        start, end, strand_col, attrs = int(cols[3]), int(cols[4]), cols[6], cols[8]

        # Parse attributes
        attr_dict = {}
        for kv in attrs.split(";"):
            parts = kv.strip().split()
            if len(parts) == 2:
                attr_dict[parts[0]] = parts[1].strip('"')

        if attr_dict.get("gene_name") != GENE_NAME:
            continue
        if attr_dict.get("transcript_id") != TARGET_TRANSCRIPT:
            continue

        if feature == "exon":
            exons.append((start, end))
        elif feature == "transcript":
            strand = strand_col
            tss = start if strand_col == "+" else end

if not exons or tss is None:
    raise ValueError(f"Transcript {TARGET_TRANSCRIPT} not found in GTF.")

exons = sorted(exons)
promoter_start = tss - PROMOTER_UPSTREAM if strand == "+" else tss
promoter_end   = tss if strand == "+" else tss + PROMOTER_UPSTREAM

print(f"{GENE_NAME} | Strand: {strand} | TSS: {tss} | Exons: {len(exons)}")

# Plot
probe_positions = [pos for _, pos in probes]
min_x = min(promoter_start, exons[0][0],  min(probe_positions)) - 500
max_x = max(promoter_end,   exons[-1][1], max(probe_positions)) + 500

GENE_Y = 1.0
CPG_Y  = 0.3

fig, ax = plt.subplots(figsize=(22, 7))

# CpG regions
for lbl, rs, re, col in CPG_REGIONS:
    ax.barh(CPG_Y, re - rs, left=rs, height=0.25,
            color=col, edgecolor="black", linewidth=1.0, zorder=2, alpha=0.9)
    ax.text((rs + re) / 2, CPG_Y + 0.01, lbl,
            ha="center", va="center", fontsize=11, fontweight="bold")
    for xpos, label in [(rs, str(rs)), (re, str(re))]:
        ax.text(xpos, CPG_Y - 0.16, label,
                ha="center", va="top", fontsize=9, rotation=90)

# Gene body
ax.barh(GENE_Y, exons[-1][1] - exons[0][0], left=exons[0][0],
        height=0.14, color="gray", zorder=1)

# Exons
for s, e in exons:
    ax.barh(GENE_Y, e - s, left=s, height=0.18,
            color="blue", edgecolor="black", linewidth=1.0, zorder=2)

# Promoter
ax.barh(GENE_Y, promoter_end - promoter_start, left=promoter_start,
        height=0.18, color="purple", edgecolor="black", linewidth=1.0, zorder=1)

# Probes
for probe_name, pos in probes:
    label = f"{MS_LABELS[probe_name]}\n{probe_name}"
    ax.vlines(pos, CPG_Y + 0.14, GENE_Y + 0.75, color="red", linewidth=2.0, zorder=3)
    ax.text(pos, GENE_Y + 0.77, label,
            rotation=90, fontsize=10, va="bottom", ha="center", color="red")

# Formatting
ax.set_xlabel("Genomic position (hg19)", fontsize=13)
ax.set_title(
    f"{GENE_NAME} — Exons, Promoter, Methylation Probes & CpG Island Regions",
    pad=80, fontsize=15,
)
ax.set_xlim(min_x, max_x)
ax.set_ylim(-0.3, 2.6)
ax.get_yaxis().set_visible(False)
for spine in ["left", "top", "right"]:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis="x", labelsize=11)

# Legend
legend_elements = [
    Patch(facecolor="blue",    edgecolor="black", label="Exon"),
    Patch(facecolor="gray",    edgecolor="black", label="Intron"),
    Patch(facecolor="purple",  edgecolor="black", label=f"Promoter ({PROMOTER_UPSTREAM} bp upstream)"),
    Patch(facecolor="red",     edgecolor="black", label="Methylation probe"),
    Patch(facecolor="#2ca02c", edgecolor="black", label="CpG Island"),
    Patch(facecolor="#17becf", edgecolor="black", label="N Shore"),
    Patch(facecolor="#9edae5", edgecolor="black", label="S Shore"),
    Patch(facecolor="#ffbb78", edgecolor="black", label="S Shelf"),
]
ax.legend(handles=legend_elements, loc="lower center",
          bbox_to_anchor=(0.5, -0.18), ncol=4, frameon=False,
          fontsize=11, title="Features", title_fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_FIG, dpi=200, bbox_inches="tight")
print(f"Figure saved to: {OUTPUT_FIG}")
plt.show()

In [ ]:
# Merging datasets

In [ ]:
import pandas as pd
BASE_DIR = r"C:\Users\jlapi\Desktop\XENA2"
CANCER_TYPES = ["BRCA"]
ENSEMBL_ID = "ENSG00000107159.13"
METHYLATION_PROBES = [
    "cg19257550", "cg20610181", "cg06908460",
    "cg13849253", "cg09566069", "cg14563831", "cg13938361"
]

# PIPELINE
for cancer in CANCER_TYPES:
    print(f"\nProcessing {cancer}...")
    d = f"{BASE_DIR}\\{cancer}\\TCGA-{cancer}"

# Step 1: Extract CA9 expression and transpose
    df_expr = pd.read_csv(f"{d}.star_tpm.tsv", sep="\t", low_memory=False)
    df_expr = (
        df_expr[df_expr["Ensembl_ID"] == ENSEMBL_ID]
        .set_index("Ensembl_ID")
        .T
        .rename(columns={ENSEMBL_ID: "CA9_Expression"})
        .rename_axis("sample")
        .reset_index()
    )

# Step 2: Merge with clinical data
    df_clinical = pd.read_csv(f"{d}.clinical.tsv", sep="\t")
    df_merged = df_expr.merge(df_clinical, on="sample")

# Step 3: Merge with survival data
    df_survival = pd.read_csv(f"{d}.survival.tsv", sep="\t")
    df_merged = df_merged.merge(df_survival, on="sample")

# Step 4: Extract and transpose methylation probes
    df_meth = pd.read_csv(f"{d}.methylation450.tsv", sep="\t")
    df_meth = (
        df_meth[df_meth["Composite Element REF"].isin(METHYLATION_PROBES)]
        .set_index("Composite Element REF")
        .T
        .rename_axis("sample")
        .reset_index()
    )

# Step 5: Merge with methylation data and save
    df_final = df_merged.merge(df_meth, on="sample")
    output_path = f"{BASE_DIR}\\{cancer}\\{cancer}_final_merged.xlsx"
    df_final.to_excel(output_path, index=False)

    print(f"  Saved → {output_path}")
    print(df_final.head())

print("\nAll cancer types processed successfully.")

In [ ]:
# Gaining other genes and matching datasets

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# ── Configuration ─────────────────────────────────────────────────────────────

genes = {
    "CA9":    "ENSG00000107159",
    "HIF1A":  "ENSG00000100644",
    "EPAS1":  "ENSG00000116016",
    "EGLN1":  "ENSG00000117594",
    "VHL":    "ENSG00000134086",
    "VEGFA":  "ENSG00000112715",
    "KDR":    "ENSG00000128052",
    "SLC2A1": "ENSG00000117394",
    "LDHA":   "ENSG00000134333",
    "PFKFB3": "ENSG00000142513",
    "TPI1":   "ENSG00000111669",
    "CPT1A":  "ENSG00000110090",
    "MTHFR":  "ENSG00000177000",
    "MYC":    "ENSG00000136997",
    "KRAS":   "ENSG00000133703",
    "BRAF":   "ENSG00000157764",
    "EGFR":   "ENSG00000146648",
    "ERBB2":  "ENSG00000141736",
    "MET":    "ENSG00000105976",
    "PIK3CA": "ENSG00000121879",
    "AKT1":   "ENSG00000142208",
    "MTOR":   "ENSG00000198793",
    "FOXM1":  "ENSG00000111206",
    "PTEN":   "ENSG00000171862",
    "STK11":  "ENSG00000118046",
    "TP53":   "ENSG00000141510",
    "RB1":    "ENSG00000139687",
    "CDKN1A": "ENSG00000124762",
    "CDKN2A": "ENSG00000147889",
    "CIC":    "ENSG00000079432",
    "BAP1":   "ENSG00000163930",
    "BRCA1":  "ENSG00000012048",
    "BRCA2":  "ENSG00000139618",
    "RAD51":  "ENSG00000051180",
    "ATM":    "ENSG00000149311",
    "FANCA":  "ENSG00000187741",
    "MLH1":   "ENSG00000076242",
    "MSH2":   "ENSG00000095002",
    "MSH6":   "ENSG00000116062",
    "BCL2":   "ENSG00000171791",
    "MCL1":   "ENSG00000143384",
    "BAX":    "ENSG00000087088",
    "BAD":    "ENSG00000002330",
    "CASP3":  "ENSG00000164305",
    "CASP8":  "ENSG00000064012",
    "DNMT1":  "ENSG00000130816",
    "DNMT3A": "ENSG00000119772",
    "DNMT3B": "ENSG00000088305",
    "DNMT3L": "ENSG00000142182",
    "EZH2":   "ENSG00000106462",
    "HDAC1":  "ENSG00000116478",
    "HDAC2":  "ENSG00000196591",
    "HDAC3":  "ENSG00000171720",
    "TET1":   "ENSG00000138336",
    "TET2":   "ENSG00000168769",
    "TET3":   "ENSG00000187605",
    "TDG":    "ENSG00000139372",
    "PBRM1":  "ENSG00000163171",
    "SETD2":  "ENSG00000181555",
    "KDM5C":  "ENSG00000126012",
    "TTN":    "ENSG00000155657",
    "NFE2L2": "ENSG00000116044",
    "CYP1A1": "ENSG00000140505",
    "APC":    "ENSG00000134982",
    "GSK3B":  "ENSG00000082701",
    "CTNNB1": "ENSG00000168036",
    "TGFB1":  "ENSG00000105329",
    "SNAI1":  "ENSG00000185591",
    "SNAI2":  "ENSG00000019549",
    "NOTCH1": "ENSG00000148400",
    "TWIST1": "ENSG00000122691",
    "VIM":    "ENSG00000026025",
    "CDH1":   "ENSG00000039068",
    "CDH2":   "ENSG00000170558",
    "CFL1":   "ENSG00000172757",
    "MMP9":   "ENSG00000100985",
    "CD274":  "ENSG00000120217",
    "PDCD1":  "ENSG00000188389",
    "CTLA4":  "ENSG00000163599",
    "AR":     "ENSG00000169083",
    "ERG":    "ENSG00000157554",
}

meth_sites = [
    "cg06908460", "cg09566069", "cg13849253", "cg13938361",
    "cg14563831", "cg19257550", "cg20610181",
]

tpm_path   = r"C:\Users\jlapi\Desktop\XENA\KIRC\TCGA-KIRC.star_tpm.tsv"
main_path  = r"C:\Users\jlapi\Desktop\XENA\KIRC\FINAL\KIRC_final_merged.xlsx"
corr_out   = r"C:\Users\jlapi\Desktop\KIRC_Gene_Methylation_Correlations.xlsx"

# ── Step 1: Load TPM, filter genes, transpose ─────────────────────────────────

ensembl_to_gene = {v: k for k, v in genes.items()}

df_tpm = pd.read_csv(tpm_path, sep="\t", low_memory=False)
df_tpm.rename(columns={df_tpm.columns[0]: "Ensembl_ID"}, inplace=True)
df_tpm["Ensembl_ID"] = df_tpm["Ensembl_ID"].str.split(".").str[0]
df_tpm = df_tpm[df_tpm["Ensembl_ID"].isin(ensembl_to_gene)]

df_tpm = (
    df_tpm.set_index("Ensembl_ID")
    .T
    .rename(columns=ensembl_to_gene)
)
df_tpm.index.name = "sample"

print(f"TPM: {df_tpm.shape[0]} samples, {df_tpm.shape[1]} genes")

# ── Step 2: Load main file and merge ──────────────────────────────────────────

df_main = pd.read_excel(main_path)

# Rename first column to "sample" if needed
if df_main.columns[0] != "sample":
    df_main.rename(columns={df_main.columns[0]: "sample"}, inplace=True)

df_merged = pd.merge(df_main, df_tpm, on="sample", how="inner")
print(f"Merged: {df_merged.shape[0]} samples, {df_merged.shape[1]} columns")

# ── Step 3: Pearson correlations → pivot table (genes × meth sites) ───────────

genes_present = [g for g in genes if g in df_merged.columns]
meth_present  = [m for m in meth_sites if m in df_merged.columns]

print(f"Genes found: {len(genes_present)} / {len(genes)}")
print(f"Meth sites found: {len(meth_present)} / {len(meth_sites)}")

records = []
for gene in genes_present:
    for meth in meth_present:
        pair = df_merged[[gene, meth]].dropna()
        if len(pair) >= 3:
            r, p = stats.pearsonr(pair[gene], pair[meth])
        else:
            r, p = np.nan, np.nan
        records.append({"Gene": gene, "Methylation_Site": meth, "r": r, "p": p})

results_df = pd.DataFrame(records)

# Pivot: rows = genes, columns = methylation sites (matching your Excel layout)
pivot_r = results_df.pivot(index="Gene", columns="Methylation_Site", values="r")
pivot_p = results_df.pivot(index="Gene", columns="Methylation_Site", values="p")

# Preserve gene order as defined in the genes dict
pivot_r = pivot_r.reindex(index=genes_present, columns=meth_present)
pivot_p = pivot_p.reindex(index=genes_present, columns=meth_present)

# ── Step 4: Save ──────────────────────────────────────────────────────────────

with pd.ExcelWriter(corr_out, engine="openpyxl") as writer:
    pivot_r.to_excel(writer, sheet_name="Pearson_r")
    pivot_p.to_excel(writer, sheet_name="P_value")
    results_df.to_excel(writer, sheet_name="All_Pairs", index=False)

print(f"Saved → {corr_out}")